# Complete End-to-End DART+AGIL Experiment Notebook

This single notebook includes:
- Dataset download/ingestion
- Preprocessing + temporal split
- Proposed model (DART+AGIL)
- Baselines
- Ablations
- Full evaluation metrics
- Table exports
- PDF plot exports (pattern-filled, no color)


In [ ]:
import os
import sys
import json
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch

from concept_drift.utils.reproducibility import set_seed
from concept_drift.utils.io import ensure_dir, write_json
from concept_drift.data.download import ensure_dataset
from concept_drift.data.preprocess import load_nsl_kdd, load_generic_csv_dir, preprocess_dataframe
from concept_drift.data.splits import temporal_split_indices
from concept_drift.data.dataloaders import make_loader
from concept_drift.models.dart_agil import DARTAGIL
from concept_drift.models.baselines import build_baselines, IFDriftBaseline
from concept_drift.training.trainer import fit
from concept_drift.eval.metrics import classification_metrics
from concept_drift.eval.fewshot import fewshot_scores
from concept_drift.eval.obfuscation import apply_idp, apply_ibp, apply_apr, apply_inp
from concept_drift.eval.reporting import export_all_reports


In [ ]:
# Configuration (edit as needed)
cfg = {
    "seed": 42,
    "output_dir": "outputs",
    "data": {
        "datasets": ["nsl_kdd", "cicids2017", "cicddos2019"],
        "root": "data",
        "max_rows_per_dataset": None,
        "temporal_split": {"train": 0.70, "val": 0.15, "test": 0.15},
    },
    "model": {"hidden_dim": 128, "latent_dim": 64, "dropout": 0.2},
    "training": {
        "batch_size": 256,
        "epochs": 20,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "kl_weight": 0.05,
        "agil_weight": 0.5,
        "grad_clip": 5.0,
        "early_stopping_patience": 5,
    },
    "agil": {"epsilon": 0.03, "steps": 3, "step_size": 0.01},
    "ablation": {"run": True, "variants": ["full", "no_dart", "no_agil", "no_online_tl"]},
    "evaluation": {
        "run_obfuscation": True,
        "obfuscation": {"idp": [0.1, 0.2, 0.3], "ibp": [0.1, 0.3, 0.5], "apr": [0.5], "inp": [0.5]},
        "few_shot_shots": [1, 3, 5, 10],
    },
}

set_seed(cfg["seed"])
out = Path(cfg["output_dir"])
for d in ["logs", "figures", "tables", "checkpoints", "reports"]:
    ensure_dir(out / d)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
def infer_probs_torch(model, X, device):
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(X).to(device)
        logits, _, _, _ = model(xb)
        return torch.sigmoid(logits).cpu().numpy()


def evaluate_obfuscations(y_true, model_name, predict_fn, X, cfg):
    rows = []
    for p in cfg["evaluation"]["obfuscation"]["idp"]:
        rows.append({"model": model_name, "scenario": f"idp_{p}", **classification_metrics(y_true, predict_fn(apply_idp(X, p)))})
    for p in cfg["evaluation"]["obfuscation"]["ibp"]:
        rows.append({"model": model_name, "scenario": f"ibp_{p}", **classification_metrics(y_true, predict_fn(apply_ibp(X, p)))})
    for p in cfg["evaluation"]["obfuscation"]["apr"]:
        rows.append({"model": model_name, "scenario": f"apr_{p}", **classification_metrics(y_true, predict_fn(apply_apr(X, p)))})
    for p in cfg["evaluation"]["obfuscation"]["inp"]:
        rows.append({"model": model_name, "scenario": f"inp_{p}", **classification_metrics(y_true, predict_fn(apply_inp(X, p)))})
    return rows


In [ ]:
# Full pipeline execution
metrics_rows = []
fewshot_rows = []
drift_rows = []

for ds_name in cfg["data"]["datasets"]:
    ds_path = ensure_dataset(ds_name, cfg["data"]["root"])
    if ds_name == "nsl_kdd":
        df = load_nsl_kdd(ds_path)
    else:
        try:
            df = load_generic_csv_dir(ds_path)
        except Exception as exc:
            print(f"[WARN] Skipping {ds_name}: {exc}")
            continue

    X, y, _t, _features, _imputer, _scaler = preprocess_dataframe(df, cfg["data"]["max_rows_per_dataset"])
    tr, va, te = temporal_split_indices(len(y), **cfg["data"]["temporal_split"])
    X_tr, y_tr = X[tr], y[tr]
    X_va, y_va = X[va], y[va]
    X_te, y_te = X[te], y[te]

    # Baselines
    for baseline in build_baselines(seed=cfg["seed"]):
        t0 = time.perf_counter()
        baseline.fit(X_tr, y_tr)
        train_sec = time.perf_counter() - t0
        y_prob = baseline.predict_proba(X_te)
        metrics_rows.append({"dataset": ds_name, "model": baseline.name, "scenario": "clean", "train_sec": train_sec, **classification_metrics(y_te, y_prob)})

        if cfg["evaluation"]["run_obfuscation"]:
            metrics_rows.extend([{"dataset": ds_name, **r} for r in evaluate_obfuscations(y_te, baseline.name, baseline.predict_proba, X_te, cfg)])

        fs = fewshot_scores(baseline.model, X_tr, y_tr, X_te, y_te, cfg["evaluation"]["few_shot_shots"], repeats=20)
        for item in fs:
            fewshot_rows.append({"dataset": ds_name, "model": baseline.name, **item})

    # IF-DR
    ifdr = IFDriftBaseline()
    ifdr.fit(X_tr, y_tr)
    y_prob = ifdr.predict_proba(X_te)
    drift_rows.append({"dataset": ds_name, "model": "ifdr", "drift_rate": ifdr.drift_rate(X_te), **classification_metrics(y_te, y_prob)})

    # Proposed + ablations
    train_loader = make_loader(X_tr, y_tr, cfg["training"]["batch_size"], shuffle=True)
    val_loader = make_loader(X_va, y_va, cfg["training"]["batch_size"], shuffle=False)

    variants = cfg["ablation"]["variants"] if cfg["ablation"]["run"] else ["full"]
    for variant in variants:
        run_cfg = json.loads(json.dumps(cfg))
        run_cfg["variant"] = variant
        if variant == "no_dart":
            run_cfg["training"]["kl_weight"] = 0.0
        if variant == "no_agil":
            run_cfg["training"]["agil_weight"] = 0.0

        model = DARTAGIL(
            input_dim=X.shape[1],
            hidden_dim=cfg["model"]["hidden_dim"],
            latent_dim=cfg["model"]["latent_dim"],
            dropout=cfg["model"]["dropout"],
        ).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=run_cfg["training"]["lr"], weight_decay=run_cfg["training"]["weight_decay"])
        ckpt = out / "checkpoints" / f"{ds_name}_{variant}.pt"

        t0 = time.perf_counter()
        fit(model, train_loader, val_loader, opt, device, run_cfg, ckpt)
        train_sec = time.perf_counter() - t0

        y_prob = infer_probs_torch(model, X_te, device)
        metrics_rows.append({"dataset": ds_name, "model": f"dart_agil_{variant}", "scenario": "clean", "train_sec": train_sec, **classification_metrics(y_te, y_prob)})

        if cfg["evaluation"]["run_obfuscation"]:
            pred_fn = lambda Xin: infer_probs_torch(model, Xin, device)
            metrics_rows.extend([{"dataset": ds_name, **r} for r in evaluate_obfuscations(y_te, f"dart_agil_{variant}", pred_fn, X_te, cfg)])

if not metrics_rows:
    raise RuntimeError("No dataset loaded. Place dataset CSV files as needed and rerun.")

metrics_df = pd.DataFrame(metrics_rows)
fewshot_df = pd.DataFrame(fewshot_rows) if fewshot_rows else None
drift_df = pd.DataFrame(drift_rows) if drift_rows else None

metrics_df.to_csv(out / "tables" / "metrics_all.csv", index=False)
if fewshot_df is not None:
    fewshot_df.to_csv(out / "tables" / "fewshot.csv", index=False)
if drift_df is not None:
    drift_df.to_csv(out / "tables" / "drift.csv", index=False)

export_all_reports(metrics_df=metrics_df, fewshot_df=fewshot_df, drift_df=drift_df, out_dir=out)

write_json(out / "logs" / "run_summary.json", {
    "datasets_processed": sorted(metrics_df["dataset"].unique().tolist()),
    "rows_metrics": int(len(metrics_df)),
    "rows_fewshot": int(len(fewshot_rows)),
    "rows_drift": int(len(drift_rows)),
})

print("Completed. Tables/plots written to", out)


In [ ]:
# Quick inspection tables
display(pd.read_csv(out / "tables" / "table_clean_summary.csv").head())
if (out / "tables" / "table_ablation.csv").exists():
    display(pd.read_csv(out / "tables" / "table_ablation.csv").head())
if (out / "tables" / "table_obfuscation_degradation.csv").exists():
    display(pd.read_csv(out / "tables" / "table_obfuscation_degradation.csv").head())


## Expected outputs
- `outputs/tables/metrics_all.csv`
- `outputs/tables/clean_metrics.csv`
- `outputs/tables/table_clean_summary.csv`
- `outputs/tables/table_obfuscation_degradation.csv`
- `outputs/tables/table_ablation.csv`
- `outputs/tables/table_fewshot.csv` (if available)
- `outputs/tables/table_drift.csv` (if available)
- `outputs/figures/*.pdf`
